# DM_CCP — Data Mining Project
### NED University | CT-377 | Fall 2025

In [ ]:
# ============================================================
# STEP 1 — INSTALL & IMPORT ALL LIBRARIES
# ============================================================
!pip install requests beautifulsoup4 nltk textblob wordcloud scikit-learn transformers matplotlib pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import nltk
import string
import time
import random
import matplotlib
matplotlib.use("Agg")          # headless backend — works on Colab AND Jupyter
import matplotlib.pyplot as plt
from textblob import TextBlob
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from transformers import pipeline

nltk.download("stopwords", quiet=True)
nltk.download("punkt",     quiet=True)
from nltk.corpus import stopwords

print(" All libraries loaded.")

 All libraries loaded.


In [ ]:
# ============================================================
# STEP 2 — USER INPUT
# ============================================================
keyword = input("Enter keyword (e.g., Machine Learning): ").strip()
print(f"Keyword set to: {keyword}")

Enter keyword (e.g., Machine Learning): data science
Keyword set to: data science


In [ ]:
# ============================================================
# STEP 3 — DATA COLLECTION
# ============================================================

import requests, time, string, pandas as pd
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower().replace("\n", " ")
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [w for w in text.split() if w not in STOP]
    return " ".join(tokens)[:2000]

# ---------- SOURCE 1: Wikipedia ----------
def scrape_wikipedia(kw):
    results = []
    variants = [
    kw,
    kw + " algorithms",
    kw + " applications",
    kw + " techniques",
    kw + " examples"
]
    headers = {"User-Agent": "Mozilla/5.0 (educational-project)"}

    for term in variants:
        try:
            slug = term.strip().replace(" ", "_").title()
            url  = f"https://en.wikipedia.org/wiki/{slug}"
            r    = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")

            h1   = soup.find("h1")
            title = h1.text if h1 else term

            paras = soup.find_all("p")
            text  = " ".join(p.get_text() for p in paras if len(p.get_text()) > 50)

            if not text:
                continue

            results.append({
                "url": url,
                "title": title,
                "author": "Wikipedia Contributors",
                "date": "2024-01-01",
                "source": "Wikipedia",
                "raw_text": text[:3000],
                "comments": "Community-edited reference article. Neutral encyclopedic tone."
            })

            time.sleep(0.8)

        except Exception as e:
            print(f"  Wikipedia skip ({term}): {e}")

    return results


# ---------- SOURCE 2: Dev.to ----------
def scrape_devto(kw):
    results = []
    tag = kw.replace(" ", "").lower()

    try:
        r = requests.get(f"https://dev.to/api/articles?tag={tag}&per_page=30", timeout=12)

        for art in r.json()[:6]:
            try:
                detail = requests.get(f"https://dev.to/api/articles/{art['id']}", timeout=10).json()
                body   = detail.get("body_markdown") or art.get("description", "")

                results.append({
                    "url": art.get("url", ""),
                    "title": art.get("title", "No Title"),
                    "author": art.get("user", {}).get("name", "Unknown"),
                    "date": str(art.get("published_at", "Unknown"))[:10],
                    "source": "Dev.to",
                    "raw_text": body,
                    "comments": f"Reactions: {art.get('positive_reactions_count',0)}, Comments: {art.get('comments_count',0)}"
                })

                time.sleep(0.5)

            except Exception as e:
                print(f"  Dev.to article skip: {e}")

    except Exception as e:
        print(f"  Dev.to API error: {e}")

    return results


# ---------- SOURCE 3: HackerNews ----------
def scrape_hackernews(kw):
    results = []

    try:
        api = f"https://hn.algolia.com/api/v1/search?query={kw}&tags=story&hitsPerPage=35"
        r   = requests.get(api, timeout=12)

        for hit in r.json().get("hits", [])[:8]:
            story_url = hit.get("url", "")
            body = hit.get("story_text") or ""

            if story_url and not body:
                try:
                    sr = requests.get(story_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=8)
                    ss = BeautifulSoup(sr.text, "html.parser")
                    body = " ".join(p.get_text() for p in ss.find_all("p"))[:3000]
                except:
                    body = hit.get("title", "")

            results.append({
                "url": story_url or f"https://news.ycombinator.com/item?id={hit.get('objectID','')}",
                "title": hit.get("title", "No Title"),
                "author": hit.get("author", "Unknown"),
                "date": str(hit.get("created_at", "Unknown"))[:10],
                "source": "HackerNews",
                "raw_text": body if body else hit.get("title", ""),
                "comments": f"Points: {hit.get('points',0)}, Comments: {hit.get('num_comments',0)}"
            })

            time.sleep(0.4)

    except Exception as e:
        print(f"  HackerNews API error: {e}")

    return results


# ---------- RUN ----------
print(f"\n🔍 Collecting blogs for: '{keyword}'\n")

print("[1/3] Wikipedia...")
wiki_data = scrape_wikipedia(keyword)
print(f"  {len(wiki_data)} articles")

print("[2/3] Dev.to...")
devto_data = scrape_devto(keyword)
print(f"  {len(devto_data)} articles")

print("[3/3] HackerNews...")
hn_data = scrape_hackernews(keyword)
print(f"  {len(hn_data)} articles")

all_data = wiki_data + devto_data + hn_data


# ---------- FALLBACK ----------
if len(all_data) < 10:
    print("Warning: Less than 10 articles collected. Consider increasing limits.")

    all_data = [
        {"url":"fb1","title":"Intro to Machine Learning","author":"John Doe","date":"2024-01-01","source":"Backup",
         "raw_text":"Machine learning is a subset of AI that teaches computers to learn from data without explicit programming.",
         "comments":"Very well written!"},
        {"url":"fb2","title":"Big Data Explained","author":"Jane Smith","date":"2024-02-15","source":"Backup",
         "raw_text":"Big data refers to datasets too large for traditional systems.",
         "comments":"Helpful explanation."}
    ]


df = pd.DataFrame(all_data)
df["clean_text"] = df["raw_text"].apply(clean_text)

print("\nData Collection Complete!")
print(f"Total articles : {len(df)}")
print(f"Sources        : {df['source'].value_counts().to_dict()}")
print(f"DataFrame shape: {df.shape}")

df[["title","author","date","source"]].head(15)


🔍 Collecting blogs for: 'data science'

[1/3] Wikipedia...
  1 articles
[2/3] Dev.to...
  6 articles
[3/3] HackerNews...
  8 articles

Data Collection Complete!
Total articles : 15
Sources        : {'HackerNews': 8, 'Dev.to': 6, 'Wikipedia': 1}
DataFrame shape: (15, 8)


,title,author,date,source
0,Data science,Wikipedia Contributors,2024-01-01,Wikipedia
1,Case Study: Reducing Data Ingestion Latency by...,NARESH-CN2,2026-04-29,Dev.to
2,What 500 curated failure pairs actually fix: a...,namakoo [IDFU],2026-04-29,Dev.to
3,Why ChatGPT will silently lie about your bank ...,Kyr,2026-04-29,Dev.to
4,Predicting Telecom Customer Churn with scikit-...,Tebogo Tseka,2026-04-29,Dev.to
5,Introduction to Python for Data Analytics,Timothy Njeru,2026-04-29,Dev.to
6,Coletei 8.571 queries em sete dias e descobri ...,Alexandre Caramaschi,2026-04-29,Dev.to
7,"Goodbye, data science",sonabinu,2022-11-29,HackerNews
8,Berkeley offers its data science course online...,seycombi,2018-04-05,HackerNews
9,Show HN: I wrote a book about using data scien...,andrewnc,2021-02-24,HackerNews


In [ ]:
# ============================================================
# STEP 4 — NLP ANALYSIS
# Keyword extraction · Topic modeling · Tone · Motive
# ============================================================

# --- TF-IDF Keyword Extraction ---
vectorizer = TfidfVectorizer(stop_words="english", max_features=50)
X = vectorizer.fit_transform(df["clean_text"])
features = vectorizer.get_feature_names_out()

def top_keywords(row):
    scores = sorted(zip(features, row), key=lambda x: x[1], reverse=True)
    return [w for w, _ in scores[:5]]

df["keywords"] = [top_keywords(row) for row in X.toarray()]

# --- LDA Topic Modeling ---
n_topics = min(3, len(df))
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(X)
topics = [[features[i] for i in topic.argsort()[-5:]] for topic in lda.components_]
print(" Discovered Topics:")
for i, t in enumerate(topics):
    print(f"  Topic {i+1}: {t}")

# --- Motive Detection ---
def detect_motive(text):
    t = text.lower()
    if any(p in t for p in ["how to", "step by step", "tutorial", "guide"]):
        return "Tutorial"
    elif any(p in t for p in ["buy", "pricing", "discount", "offer", "sale"]):
        return "Promotional"
    elif any(p in t for p in ["research", "study", "findings", "paper", "journal"]):
        return "Academic"
    elif any(p in t for p in ["i think", "in my opinion", "i believe", "personally"]):
        return "Opinion"
    return "Informative"

# --- Tone Detection (Subjectivity via TextBlob) ---
def detect_tone(text):
    subjectivity = TextBlob(str(text)).sentiment.subjectivity
    return "Subjective/Opinion" if subjectivity > 0.5 else "Objective/Factual"

df["motive"] = df["raw_text"].apply(detect_motive)
df["tone"]   = df["raw_text"].apply(detect_tone)

print("\n Motive Distribution:", df["motive"].value_counts().to_dict())
print(" Tone Distribution:  ", df["tone"].value_counts().to_dict())
df[["title","source","motive","tone","keywords"]].head(12)

 Discovered Topics:
  Topic 1: ['churn', 'uma', 'book', 'python', 'data']
  Topic 2: ['compute', 'statement', 'ingestion', 'metaflow', 'seeds']
  Topic 3: ['information', 'using', 'page', 'data', 'science']

 Motive Distribution: {'Academic': 4, 'Informative': 4, 'Tutorial': 4, 'Promotional': 2, 'Opinion': 1}
 Tone Distribution:   {'Objective/Factual': 11, 'Subjective/Opinion': 4}


,title,source,motive,tone,keywords
0,Data science,Wikipedia,Academic,Objective/Factual,"[science, data, information, statistics, compu..."
1,Case Study: Reducing Data Ingestion Latency by...,Dev.to,Informative,Objective/Factual,"[ingestion, compute, like, run, python]"
2,What 500 curated failure pairs actually fix: a...,Dev.to,Academic,Objective/Factual,"[seeds, model, text, vs, data]"
3,Why ChatGPT will silently lie about your bank ...,Dev.to,Tutorial,Objective/Factual,"[statement, python, page, single, real]"
4,Predicting Telecom Customer Churn with scikit-...,Dev.to,Tutorial,Subjective/Opinion,"[churn, telecom, customer, customers, pipeline]"
5,Introduction to Python for Data Analytics,Dev.to,Tutorial,Subjective/Opinion,"[python, jupyter, notebook, analytics, code]"
6,Coletei 8.571 queries em sete dias e descobri ...,Dev.to,Academic,Objective/Factual,"[uma, pipeline, analytics, book, churn]"
7,"Goodbye, data science",HackerNews,Opinion,Subjective/Opinion,"[insane, ideas, data, work, want]"
8,Berkeley offers its data science course online...,HackerNews,Informative,Objective/Factual,"[page, using, data, analytics, book]"
9,Show HN: I wrote a book about using data scien...,HackerNews,Tutorial,Subjective/Opinion,"[book, data, people, use, science]"


In [ ]:
# ============================================================
# STEP 5 — LLM COMMENT GENERATION (GPT-2, no API key needed)
# ============================================================
print("Loading GPT-2 for comment generation (first run downloads ~500 MB)...")
generator = pipeline("text-generation", model="gpt2", pad_token_id=50256)

def generate_comment(title):
    if not isinstance(title, str) or title in ("Error", "No Title"):
        return "Interesting read!"
    prompt = f"Write a short user comment about this blog post: {title[:80]}"
    try:
        res = generator(prompt, max_new_tokens=25, do_sample=True,
                        temperature=0.7, top_p=0.9, return_full_text=False)
        return res[0]["generated_text"].replace("\n", " ").strip()
    except:
        return "Great article!"

df["generated_comment"] = df["title"].apply(generate_comment)
print(" Comments generated.")
df[["title","generated_comment"]].head(10)

Loading GPT-2 for comment generation (first run downloads ~500 MB)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_c

 Comments generated.


,title,generated_comment
0,Data science,/data science is a field where you'll find man...
1,Case Study: Reducing Data Ingestion Latency by...,By using the data you can reduce data use in y...
2,What 500 curated failure pairs actually fix: a...,The next post is going to focus on the 500 fai...
3,Why ChatGPT will silently lie about your bank ...,without your permission.
4,Predicting Telecom Customer Churn with scikit-...,. I'll also write a follow up post about this...
5,Introduction to Python for Data Analytics,Introduction to Python for Data Analytics I ha...
6,Coletei 8.571 queries em sete dias e descobri ...,o. Read more at this blog post: Read more at...
7,"Goodbye, data science",". For the past two months, we've been collect..."
8,Berkeley offers its data science course online...,". It's free to register, but you must be an ac..."
9,Show HN: I wrote a book about using data scien...,. And I'm going to try to explain to you what...


In [ ]:
# ============================================================
# STEP 6 — SENTIMENT ANALYSIS ON COMMENTS
# + POPULARITY PROXY (comment count simulation)
# ============================================================

def get_sentiment(text):
    score = TextBlob(str(text)).sentiment.polarity
    if score >  0.1: return "Positive"
    if score < -0.1: return "Negative"
    return "Neutral"

# Analyse BOTH scraped comments AND generated comments
df["scraped_sentiment"]   = df["comments"].apply(get_sentiment)
df["generated_sentiment"] = df["generated_comment"].apply(get_sentiment)
df["sentiment_score"]     = df["generated_comment"].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity)

# Popularity proxy: higher absolute sentiment → more engagement
random.seed(42)
df["comment_count"] = df["sentiment_score"].apply(
    lambda x: int(abs(x) * 100) + random.randint(5, 20))

print(" Generated Comment Sentiment:", df["generated_sentiment"].value_counts().to_dict())
print(" Scraped Comment Sentiment:  ", df["scraped_sentiment"].value_counts().to_dict())
df[["title","scraped_sentiment","generated_sentiment","sentiment_score","comment_count"]].head(12)

 Generated Comment Sentiment: {'Positive': 7, 'Neutral': 7, 'Negative': 1}
 Scraped Comment Sentiment:   {'Neutral': 15}


,title,scraped_sentiment,generated_sentiment,sentiment_score,comment_count
0,Data science,Neutral,Positive,0.266667,34
1,Case Study: Reducing Data Ingestion Latency by...,Neutral,Neutral,0.000000,5
2,What 500 curated failure pairs actually fix: a...,Neutral,Negative,-0.166667,29
3,Why ChatGPT will silently lie about your bank ...,Neutral,Neutral,0.000000,12
4,Predicting Telecom Customer Churn with scikit-...,Neutral,Neutral,0.000000,12
5,Introduction to Python for Data Analytics,Neutral,Positive,0.275000,36
6,Coletei 8.571 queries em sete dias e descobri ...,Neutral,Positive,0.500000,58
7,"Goodbye, data science",Neutral,Positive,0.283333,35
8,Berkeley offers its data science course online...,Neutral,Neutral,0.044444,22
9,Show HN: I wrote a book about using data scien...,Neutral,Neutral,0.000000,6


In [ ]:
# ============================================================
# STEP 7 — VISUALIZATIONS  (5 charts + 1 word cloud)
# ============================================================
import os
os.makedirs("charts", exist_ok=True)

# ── Chart 1: Sentiment Pie Chart ──────────────────────────
fig, ax = plt.subplots(figsize=(6,6))
counts = df["generated_sentiment"].value_counts()
colors = {"Positive":"#4CAF50","Neutral":"#FFC107","Negative":"#F44336"}
c_list = [colors.get(k,"#999") for k in counts.index]
ax.pie(counts, labels=counts.index, autopct="%1.1f%%", colors=c_list, startangle=140)
ax.set_title("Sentiment Distribution of Generated Comments", fontsize=13)
plt.tight_layout()
plt.savefig("charts/chart1_sentiment_pie.png", dpi=150)
plt.show()
print("Saved chart1_sentiment_pie.png")

# ── Chart 2: Sentiment vs Popularity Scatter ─────────────
fig, ax = plt.subplots(figsize=(7,5))
color_map = {"Positive":"green","Neutral":"orange","Negative":"red"}
for sent, grp in df.groupby("generated_sentiment"):
    ax.scatter(grp["sentiment_score"], grp["comment_count"],
               label=sent, color=color_map.get(sent,"gray"), s=80, alpha=0.8)
ax.set_xlabel("Sentiment Score")
ax.set_ylabel("Comment Count (Popularity Proxy)")
ax.set_title("Sentiment Score vs Popularity")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("charts/chart2_sentiment_vs_popularity.png", dpi=150)
plt.show()
print("Saved chart2_sentiment_vs_popularity.png")

# ── Chart 3: Motive Distribution Bar Chart ───────────────
fig, ax = plt.subplots(figsize=(7,4))
motive_counts = df["motive"].value_counts()
motive_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Blog Motive Distribution")
ax.set_xlabel("Motive")
ax.set_ylabel("Number of Blogs")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("charts/chart3_motive_bar.png", dpi=150)
plt.show()
print("Saved chart3_motive_bar.png")

# ── Chart 4: Tone Distribution Bar Chart ─────────────────
fig, ax = plt.subplots(figsize=(6,4))
df["tone"].value_counts().plot(kind="bar", ax=ax, color=["#2196F3","#FF9800"], edgecolor="white")
ax.set_title("Tone Comparison Across Sources")
ax.set_xlabel("Tone")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("charts/chart4_tone_bar.png", dpi=150)
plt.show()
print("Saved chart4_tone_bar.png")

# ── Chart 5: Top Keywords Bar Chart ─────────────────────
from collections import Counter
all_kw = [kw for kws in df["keywords"] for kw in kws]
top_kw = Counter(all_kw).most_common(10)
kw_words, kw_freq = zip(*top_kw)
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(kw_words, kw_freq, color="mediumpurple")
ax.set_title("Top 10 Keywords Across All Blogs")
ax.set_xlabel("Frequency")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("charts/chart5_top_keywords.png", dpi=150)
plt.show()
print("Saved chart5_top_keywords.png")

# ── Word Cloud ────────────────────────────────────────────
all_text = " ".join(df["clean_text"].dropna())
wc = WordCloud(width=800, height=400, background_color="white",
               colormap="viridis", max_words=100).generate(all_text)
fig, ax = plt.subplots(figsize=(12,5))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud — Most Common Blog Keywords", fontsize=14)
plt.tight_layout()
plt.savefig("charts/wordcloud.png", dpi=150)
plt.show()
print("Saved wordcloud.png")

print("\n All 5 charts + word cloud saved in /charts/")

Saved chart1_sentiment_pie.png
Saved chart2_sentiment_vs_popularity.png
Saved chart3_motive_bar.png
Saved chart4_tone_bar.png
Saved chart5_top_keywords.png
Saved wordcloud.png

 All 5 charts + word cloud saved in /charts/


In [ ]:
# ============================================================
# STEP 8 — FINAL SUMMARY TABLE
# ============================================================
print("\n===== PROJECT SUMMARY =====")
print(f"Keyword          : {keyword}")
print(f"Total Blogs      : {len(df)}")
print(f"Sources          : {df["source"].value_counts().to_dict()}")
print(f"Motives Found    : {df["motive"].value_counts().to_dict()}")
print(f"Tones Found      : {df["tone"].value_counts().to_dict()}")
print(f"Sentiment (gen.) : {df["generated_sentiment"].value_counts().to_dict()}")
print(f"Avg Popularity   : {df["comment_count"].mean():.1f} (simulated)")
print("\nSample Generated Comments:")
for _, row in df[["title","generated_comment"]].head(5).iterrows():
    print(f"  [{row["title"][:40]}] → {row["generated_comment"]}")
print("\n Pipeline complete!")


===== PROJECT SUMMARY =====
Keyword          : data science
Total Blogs      : 15
Sources          : {'HackerNews': 8, 'Dev.to': 6, 'Wikipedia': 1}
Motives Found    : {'Academic': 4, 'Informative': 4, 'Tutorial': 4, 'Promotional': 2, 'Opinion': 1}
Tones Found      : {'Objective/Factual': 11, 'Subjective/Opinion': 4}
Sentiment (gen.) : {'Positive': 7, 'Neutral': 7, 'Negative': 1}
Avg Popularity   : 30.9 (simulated)

Sample Generated Comments:
  [Data science] → /data science is a field where you'll find many people with the same interests. It's an exciting field, and there
  [Case Study: Reducing Data Ingestion Late] → By using the data you can reduce data use in your business.  This blog post is about reducing data ingestion
  [What 500 curated failure pairs actually ] → The next post is going to focus on the 500 failed failures of the blog post. The next post is going to
  [Why ChatGPT will silently lie about your] → without your permission.
  [Predicting Telecom Customer Churn with 